# Tabularium, iteración 0: medir el corpus

Notebook de exploración, no es código del producto. Baja (una sola vez, con pausas) los documentos fuente a `cache/`, los parte en artículos y reglas, y mide lo que piden los pasos 2, 3 y 5 de `steps.md`: formato, línea base al 2024-01-01 y tamaño del corpus.

La lógica vive en `descarga.py`, `medicion.py` y `tablas.py`. Las fichas de cada fuente están en `docs/fuentes.md`.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

import descarga
import tablas

pd.set_option('display.max_colwidth', 120)

## $\color{#dda}{\text{1. Descarga con caché}}$

Un solo cliente, 2.5 s entre peticiones al mismo host y User-Agent genérico. Lo que ya está en `cache/` no se vuelve a pedir.

In [2]:
bajados = pd.DataFrame([descarga.get(u) for u in tablas.urls()])
bajados['MB'] = (bajados['bytes'] / 1e6).round(1)
bajados['archivo'] = bajados['url'].str.rsplit('/', n=1).str[-1].str[:60]
bajados[['archivo', 'status', 'MB', 'last_modified', 'cache']]

,archivo,status,MB,last_modified,cache
0,LIVA.doc,200,0.5,"Thu, 18 Nov 2021 18:29:23 GMT",True
1,LISR.doc,200,2.2,"Tue, 07 May 2024 00:11:29 GMT",True
2,LISR.doc,200,2.2,,True
3,CFF.doc,200,1.8,"Sat, 18 Apr 2026 02:08:56 GMT",True
4,CFF.doc,200,1.7,,True
5,RMF_2024-29122023.pdf,200,8.5,"Fri, 14 Aug 2026 01:51:06 GMT",True
6,Compilado_Segunda_Modificacion_de_la_Resolucion_Miscelanea_F,200,170.8,"Mon, 21 Jul 2025 16:04:07 GMT",True
7,RMF_2025-30122024.pdf,200,4.5,"Fri, 14 Aug 2026 01:50:51 GMT",True
8,Compilado_6a_Modificacion_RMF2025-17122025.pdf,200,8.8,"Fri, 14 Aug 2026 01:54:01 GMT",True
9,RMF_2026-DOF-28122025.pdf,200,4.7,"Thu, 05 Nov 2026 23:26:34 GMT",True


## $\color{#dda}{\text{2. Leyes: artículos y tamaño}}$

Texto del `.doc` de LeyesBiblio, un párrafo por línea. Las notas editoriales de la Cámara ("Párrafo reformado DOF ...") se separan del texto de la norma.

In [3]:
seg_leyes = tablas.leyes(descarga.ruta_cache)
t_leyes = tablas.tabla_leyes(seg_leyes)
pd.DataFrame(t_leyes)

,ley,artículos,derogados,notas de reforma,caracteres del cuerpo,caracteres de transitorios,mediana por artículo,artículo más largo
0,LIVA,79,15,274,151138,170869,1122,17526
1,LISR,236,12,285,890685,173671,2623,36858
2,CFF,421,84,453,783201,309601,932,20773


El mismo texto partido desde el PDF: cuántos artículos salen distintos por los renglones cortados (R-04).

In [4]:
pd.DataFrame(tablas.tabla_pdf_contra_doc(descarga.ruta_cache))

,ley,artículos (.doc),artículos (PDF),con texto distinto,solo en uno de los dos
0,LIVA,79,79,5,0
1,LISR,236,236,23,0
2,CFF,421,421,6,0


## $\color{#dda}{\text{3. Línea base al 2024-01-01 (R-02)}}$

Se cuentan los artículos que cambiaron después del corte de dos formas: por las notas de reforma del texto vigente y por diff contra una copia fechada (archive.org). Deben coincidir.

In [5]:
t_base = tablas.tabla_linea_base(seg_leyes)
pd.DataFrame(t_base).T

,0,1,2
ley,LIVA,LISR,CFF
artículos,79,236,421
con nota de cambio posterior al corte,0,1,54
distintos de la línea base,0,1,54
nuevos,0,0,4
solo compilación de cantidades,0,0,10
coinciden nota y diff,True,True,True
publicaciones del DOF,0,1,6
caracteres de las versiones base que cambian,0,10911,198202


In [6]:
pd.DataFrame(tablas.tabla_decretos(seg_leyes))

,ley,DOF,artículos,cuáles
0,LISR,2024-04-01,1,79
1,CFF,2024-12-30,2,"32-A, 32-H"
2,CFF,2025-10-16,1,124
3,CFF,2025-11-07,32,"103, 104, 105, 113 Bis, 115 Ter, 141, 143, 144, 145, 150, 151, 156-Bis, 156-Ter, 17-F, 17-H, 17-H Bis, 27, 29-A, 29-..."
4,CFF,2025-11-14,1,130
5,CFF,2025-12-28,24,"102, 104, 108, 112, 115, 150, 32-A, 32-H, 80, 82, 84, 84-B, 84-D, 84-F, 84-H, 84-J, 84-L, 86, 86-B, 86-F, 88, 90, 90..."
6,CFF,2026-04-09,1,141


## $\color{#dda}{\text{4. RMF: reglas y modificaciones}}$

PDF del minisitio del SAT. La versión compilada marca cada regla reformada, adicionada o derogada con la modificación y su fecha de publicación.

In [7]:
seg_rmf = tablas.rmf(descarga.ruta_cache)
t_rmf = tablas.tabla_rmf(seg_rmf)
pd.DataFrame(t_rmf)

,RMF,reglas (original),reglas (última compilada),reglas con marca de cambio,caracteres de las reglas,caracteres de transitorios,mediana por regla,regla más larga
0,2024,1122,1110,49,2348065,28494,1282,59000
1,2025,1150,1173,128,2405180,21709,1227,61264
2,2026,1184,1191,38,2489896,27883,1261,65197


In [8]:
pd.DataFrame(tablas.tabla_modificaciones(seg_rmf)).pivot_table(
    index=['RMF', 'modificación', 'DOF'], columns='efecto', values='reglas', fill_value=0, aggfunc='sum')

efecto                                     adicionada  derogada  reformada
RMF  modificación DOF                                                     
2024 primera      4 de junio de 2024                4         0         14
     segunda      11 de octubre de 2024             3         1         28
2025 cuarta       09 de julio de 2025               2         0          1
     primera      22 de enero de 2025               0         6          1
     quinta       22 de octubre de 2025            12         5         54
     segunda      07 de abril de 2025               1         0         12
     sexta        17 de diciembre de 2025           1         0         17
     tercera      13 de mayo de 2025               11         1         24
2026 primera      09 de julio de 2026               8         1         29
     segunda      01 de octubre de 2026             0         0          2

## $\color{#dda}{\text{5. Corpus total frente a la estimación}}$

La sección 10 de `plan.md` estimó unos 20 MB de texto y menos de 3 GB en total.

In [9]:
t_anexos = tablas.tabla_anexos(descarga.ruta_cache)
pd.DataFrame(t_anexos)

,documento,páginas,páginas sin texto,caracteres
0,"Anexo 20, estándar CFDI 4.0",131,6,210140
1,"Anexo 20, guía de llenado",123,0,183081
2,Anexo 29 de la RMF 2026,22,0,76127
3,Anexo 5 de la RMF 2026 (cantidades del CFF),11,0,45711
4,Anexo 8 de la RMF 2026 (tarifas del ISR),19,0,40224


In [10]:
pd.DataFrame(tablas.tabla_corpus(t_leyes, t_base, t_rmf, t_anexos))

,parte,caracteres,MB aprox.
0,Leyes vigentes (cuerpo y transitorios),2479165,2.55
1,Versiones base de los artículos reformados,209113,0.22
2,RMF 2024 a 2026 (última compilada de cada año),7321227,7.54
3,"Anexos (20, 29, 5 y 8)",555283,0.57
4,Total,10564788,10.88


In [11]:
articulos = sum(f['artículos'] for f in t_leyes)
reglas = sum(f['reglas (última compilada)'] for f in t_rmf)
cache_mb = sum(f.stat().st_size for f in descarga.CACHE.rglob('*') if f.is_file()) / 1e6
print(f'unidades: {articulos} artículos y {reglas} reglas (suma de los tres años)')
print(f'caché de documentos fuente: {cache_mb:.0f} MB')

unidades: 736 artículos y 3474 reglas (suma de los tres años)
caché de documentos fuente: 289 MB


## $\color{#dda}{\text{6. Largo de las unidades}}$

Sirve para la pregunta abierta de la iteración 2: embeber por artículo completo, por fracción o con ventanas.

In [12]:
largos = pd.DataFrame(tablas.largos_unidades(seg_leyes, seg_rmf))
resumen = largos.groupby('documento')['caracteres'].agg(
    unidades='count', mediana='median', p90=lambda s: s.quantile(0.9), maximo='max',
    sobre_2000=lambda s: (s > 2000).mean().round(2), sobre_30000=lambda s: (s > 30000).sum())
resumen

,unidades,mediana,p90,maximo,sobre_2000,sobre_30000
documento,,,,,,
CFF,421,932.0,4601.0,20773,0.26,0
LISR,236,2595.0,7548.0,36858,0.59,1
LIVA,79,1122.0,5197.4,17526,0.30,0
RMF 2026,1191,1261.0,4538.0,65197,0.32,3


In [13]:
tablas.figura_largos(largos.to_dict('records'))